In [3]:
import pandas as pd
import re


# -------------------------
# CLEANING FUNCTION
# -------------------------
def clean_caption(caption: str, stopwords: set[str], drop_single_chars: bool = True) -> str:
    """
    Pulisce una caption:
    - lowercase
    - rimozione punteggiatura
    - rimozione stopwords
    - opzionale: rimozione token di lunghezza 1
    """

    caption = caption.lower()
    caption = re.sub(r"[^\w\s]", "", caption)

    words = caption.split()

    cleaned = []
    for w in words:
        if w in stopwords:
            continue
        if drop_single_chars and len(w) == 1:
            continue
        cleaned.append(w)

    return " ".join(cleaned)


# -------------------------
# PARSING FUNCTION
# -------------------------
def load_flickr_token_file(path: str):
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["image", "caption"]
    )

    images = []
    captions = []

    for _, row in df.iterrows():
        image = row["image"].split("#")[0].rsplit(".", 1)[0]
        images.append(image)
        captions.append(row["caption"])

    return images, captions


# -------------------------
# MAIN PIPELINE
# -------------------------
stopwords = {
    "the", "a", "an", 
    "is", "are", 
    "his", "her", "their", "its"
}

images, raw_captions = load_flickr_token_file(
    "../Flickr8k/Flickr8k_text/Flickr8k.token.txt"
)

cleaned_captions = [
    clean_caption(c, stopwords, drop_single_chars=True)
    for c in raw_captions
]

df = pd.DataFrame({
    "image": images,
    "caption": cleaned_captions
})

df.to_csv("../data/captions/captions.csv", index=False)

print(df.head())
print("Unique images:", df["image"].nunique())
print("Total captions:", len(df))

                   image                                            caption
0  1000268201_693b08cb0e  child in pink dress climbing up set of stairs ...
1  1000268201_693b08cb0e                    girl going into wooden building
2  1000268201_693b08cb0e         little girl climbing into wooden playhouse
3  1000268201_693b08cb0e           little girl climbing stairs to playhouse
4  1000268201_693b08cb0e  little girl in pink dress going into wooden cabin
Unique images: 8092
Total captions: 40460
